### Linear Algebra for Neural Networks

The math underneath every `W @ x` in this series. Vectors, matrices, matrix multiplication, inverse, eigenvalues/eigenvectors (fuller treatment in `unsupervised.ipynb`'s PCA section, brief version here), norms (tie directly to `regularization.ipynb`'s L1/L2 penalties).

#### 0. Vectors and basic operations

A vector is an ordered list of numbers, e.g. a 2-feature document representation x = [0.6, 0.7] (mentions_IRS-strength, urgency-strength, toy values).
```
addition:       [1,2] + [3,4] = [4,6]           (element-wise)
scalar multiply: 2 * [1,2] = [2,4]               (every element scaled)
dot product:    [1,2] . [3,4] = 1*3 + 2*4 = 11   (sum of element-wise products, a single number)
```
The dot product is the single most-used operation in this entire series, it is literally what `w . x` computes in every logistic regression score, every attention score in `llm-mechanics/llm-architecture.ipynb`, every leaf-split decision under the hood.

In [ ]:
import numpy as np

a = np.array([1, 2])
b = np.array([3, 4])

print("addition:", a + b)
print("scalar multiply:", 2 * a)
print("dot product:", np.dot(a, b), "(or a @ b:", a @ b, ")")

#### 1. Matrix multiplication, worked by hand

A matrix is a grid of numbers, shape (rows, columns). Matrix multiplication C = A @ B: entry C[i,j] is the dot product of A's row i and B's column j. Requires A's column count to equal B's row count.

Worked example, same shape logic as `classical-ml.ipynb`'s multinomial logreg forward pass: X (2 docs, 3 words) @ W^T (3 words, 2 classes) -> Z (2 docs, 2 classes).
```
X = [[1, 0, 1],      W = [[2, 0, 1],     W^T = [[2, 1],
     [0, 1, 1]]           [1, 1, 0]]            [0, 1],
                                                  [1, 0]]

Z[0,0] = X_row0 . W^T_col0 = [1,0,1].[2,0,1] = 1*2+0*0+1*1 = 3
Z[0,1] = X_row0 . W^T_col1 = [1,0,1].[1,1,0] = 1*1+0*1+1*0 = 1
Z[1,0] = X_row1 . W^T_col0 = [0,1,1].[2,0,1] = 0*2+1*0+1*1 = 1
Z[1,1] = X_row1 . W^T_col1 = [0,1,1].[1,1,0] = 0*1+1*1+1*0 = 1

Z = [[3, 1],
     [1, 1]]
```
Every entry is one dot product. This is literally what `X @ W.T` computes internally, one dot product per (document, class) pair, exactly the shape mechanics used throughout `classical-ml.ipynb` and `boosting.ipynb`.

In [ ]:
X = np.array([[1, 0, 1], [0, 1, 1]])
W = np.array([[2, 0, 1], [1, 1, 0]])

Z = X @ W.T
print("Z:\n", Z)
print("shapes: X", X.shape, "@ W.T", W.T.shape, "->", Z.shape)

#### 2. Transpose and identity

Transpose (A^T): flip rows and columns, A[i,j] becomes A^T[j,i]. Used constantly just to make shapes line up for multiplication (X @ W.T above needed the transpose specifically to make the inner dimensions match).

Identity matrix (I): the matrix equivalent of the number 1, ones on the diagonal, zeros elsewhere, A @ I = A for any compatible A. Used in `regularization.ipynb`'s Ridge derivation (the lambda*I term added before inverting) and shows up any time "add a small constant to the diagonal for numerical stability" is needed.

#### 3. Matrix inverse, worked by hand (2x2)

A^-1 such that A @ A^-1 = I. For a 2x2 matrix [[a,b],[c,d]], the formula is direct: A^-1 = (1/det(A)) * [[d,-b],[-c,a]], where det(A) = ad-bc.

Worked example, the matrix from `linear-regression.ipynb`'s normal equation setup, A = [[4, 2], [2, 3]]:
```
det(A) = 4*3 - 2*2 = 12-4 = 8
A^-1 = (1/8) * [[3, -2], [-2, 4]] = [[0.375, -0.25], [-0.25, 0.5]]
```
This is exactly the operation `np.linalg.inv(X_design.T @ X_design)` performs inside linear regression's normal equation `w = (X^T X)^-1 X^T y`, solving for the weights directly without any iterative gradient descent. If det(A)=0, the matrix has NO inverse (singular), which is exactly what happens under severe multicollinearity (near-duplicate features), flagged as an assumption violation in `classical-ml.ipynb`'s Linear Regression section.

In [ ]:
A = np.array([[4, 2], [2, 3]])
det_A = np.linalg.det(A)
A_inv = np.linalg.inv(A)

print("det(A):", det_A)
print("A inverse:\n", A_inv)
print("check, A @ A_inv (should be identity):\n", np.round(A @ A_inv, 6))

#### 4. Eigenvalues and eigenvectors, brief (fuller worked treatment in `unsupervised.ipynb`'s PCA section)

For a square matrix A, an eigenvector v and eigenvalue lambda satisfy A @ v = lambda * v, applying A to v just scales it, does not change its direction. PCA's entire mechanism (finding the directions of maximum variance) is exactly finding the eigenvectors of the covariance matrix, the eigenvector with the largest eigenvalue IS the direction PC1 points in, already worked through with real numbers in `unsupervised.ipynb`, not repeated here.

#### 5. Vector norms, and the direct tie to L1/L2 regularization

L2 norm: ||v||_2 = sqrt(sum(v_i^2)), the ordinary (Euclidean) length of a vector. This is EXACTLY the sum(w^2) term in Ridge regression's penalty (see `regularization.ipynb`), Ridge's penalty is literally lambda times the squared L2 norm of the weight vector.

L1 norm: ||v||_1 = sum(|v_i|), the sum of absolute values. This is EXACTLY Lasso's penalty term, lambda times the L1 norm of the weight vector.

Worked example, w = [0.3, 0.8] (the same toy weights from `regularization.ipynb`'s L1/L2 worked comparison):
```
||w||_2 = sqrt(0.3^2 + 0.8^2) = sqrt(0.09+0.64) = sqrt(0.73) = 0.854
||w||_1 = |0.3| + |0.8| = 1.1
```
The geometric difference between these two norms (L2's circular "equal distance" contour vs. L1's diamond-shaped contour) is the actual geometric reason Lasso can produce exact zeros and Ridge cannot, the diamond shape's corners land exactly on the axes, where one coordinate is exactly zero.

In [ ]:
w = np.array([0.3, 0.8])

print("L2 norm:", np.linalg.norm(w, ord=2))
print("L1 norm:", np.linalg.norm(w, ord=1))